# Contraste estadístico de patrones de oferta de Airbnb

Este notebook continúa el análisis exploratorio realizado sobre 220.031 anuncios
de Londres, Madrid, Milán, Nueva York, Sídney y Tokio.

El EDA permitió describir diferencias entre ciudades, tipos de alojamiento y
restricciones de estancia. En esta fase se aplican métodos de estadística
inferencial para evaluar si dos patrones previamente definidos presentan evidencia
estadística y cuál es su magnitud.

Las pruebas buscan apoyar la priorización de análisis para Airbnb. No permiten
demostrar causalidad, demanda, ocupación, reservas ni rentabilidad.

## 1. De la estadística descriptiva a la inferencial

La estadística descriptiva resume lo que aparece en los datos mediante conteos,
porcentajes, medianas, cuartiles, IQR y gráficos.

La estadística inferencial añade una pregunta: ¿la diferencia observada es
suficientemente clara como para rechazar una situación inicial de ausencia de
asociación o diferencia?

Para cada contraste se utilizan cuatro elementos:

1. **Hipótesis nula (H0):** representa la ausencia de asociación o diferencia.
2. **Hipótesis alternativa (H1):** representa la existencia de una asociación o
   diferencia.
3. **P-valor:** mide lo poco compatibles que serían los resultados con H0.
4. **Tamaño del efecto:** indica la magnitud y, cuando corresponde, la dirección
   de la asociación o diferencia.

Se fija antes del análisis un nivel de significación de `alpha = 0.05`.

- Si el p-valor ajustado es menor que 0.05, se rechaza H0.
- Si es igual o superior, no existe evidencia suficiente para rechazar H0.

No rechazar H0 no demuestra que H0 sea verdadera. Además, un p-valor pequeño no
implica que la diferencia sea grande o importante para negocio. Por eso siempre
se interpretará junto al tamaño del efecto.

## 2. Hipótesis predefinidas

Las hipótesis se formulan antes de calcular resultados para evitar seleccionar
únicamente aquellas que produzcan p-valores favorables.

### Hipótesis 1: ciudad y tipo de alojamiento

**Pregunta:** ¿la composición de tipos de alojamiento cambia de forma relevante
entre las seis ciudades?

- **H0:** la ciudad y el tipo de alojamiento son independientes; las ciudades
  presentan la misma distribución de tipos.
- **H1:** existe asociación entre ciudad y tipo; al menos una ciudad presenta una
  distribución diferente.
- **Método:** prueba chi-cuadrado de independencia.
- **Tamaño del efecto:** V de Cramér.

Chi-cuadrado compara los conteos observados con los que esperaríamos si las
variables fueran independientes. V de Cramér permite medir si la asociación
detectada es pequeña o grande, en una escala entre 0 y 1.

### Hipótesis 2: estancia mínima y actividad aproximada

**Pregunta:** ¿los anuncios con estancia mínima superior a ocho noches presentan
una actividad aproximada diferente dentro de cada ciudad?

Se comparan dos grupos:

- referencia: `minimum_nights <= 8`;
- interés: `minimum_nights > 8`.

Para cada ciudad:

- **H0:** ambos grupos tienen la misma distribución de actividad aproximada.
- **H1:** ambos grupos tienen distribuciones diferentes.
- **Método:** prueba bilateral U de Mann–Whitney.
- **Tamaño del efecto:** correlación biserial por rangos.
- **Multiplicidad:** corrección de Holm sobre los seis p-valores.

Mann–Whitney es adecuado porque la actividad de reseñas es muy asimétrica,
contiene numerosos ceros y no puede suponerse normal. La prueba analiza si los
valores de un grupo tienden a ocupar posiciones superiores o inferiores a los
del otro grupo.

La corrección de Holm reduce la probabilidad de declarar una diferencia por azar
al realizar seis pruebas, una por ciudad.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

ALPHA = 0.05

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

manifest_path = project_root / "data" / "manifest.csv"

print(f"Raíz del proyecto: {project_root}")
print(f"Nivel de significación: {ALPHA}")

Raíz del proyecto: C:\Users\Coder\Factoria\proyecto8-analisis de datos\project-ai-data-analyst
Nivel de significación: 0.05


## 3. Carga y control de la población

Se reutiliza la estrategia validada en el EDA: los seis archivos permanecen
separados en `data/raw/`, pero se combinan temporalmente en un único DataFrame.
La ciudad se conserva como variable derivada del manifiesto.

No se repite una exploración general de columnas porque esa tarea ya fue
completada. Solo se comprueban los controles necesarios para asegurar que las
pruebas utilizan la población prevista.

In [2]:
required_columns = [
    "id",
    "host_id",
    "room_type",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
]

city_frames = []

manifest = pd.read_csv(manifest_path)

for record in manifest.itertuples(index=False):
    city_data = pd.read_csv(
        project_root / record.relative_path,
        usecols=required_columns,
        low_memory=False,
    )

    city_data["city"] = record.city
    city_frames.append(city_data)

analysis_data = pd.concat(
    city_frames,
    ignore_index=True,
    sort=False,
)

population_checks = pd.Series(
    {
        "listing_count": len(analysis_data),
        "unique_listing_count": analysis_data["id"].nunique(),
        "city_count": analysis_data["city"].nunique(),
        "missing_city_count": analysis_data["city"].isna().sum(),
    },
    name="value",
)

assert population_checks["listing_count"] == 220_031
assert population_checks["unique_listing_count"] == 220_031
assert population_checks["city_count"] == 6
assert population_checks["missing_city_count"] == 0

population_checks.to_frame()

,value
listing_count,220031
unique_listing_count,220031
city_count,6
missing_city_count,0


## 4. Hipótesis 1: ciudad y tipo de alojamiento

### 4.1 Conteos observados

Primero se construye una tabla de contingencia. Cada celda contiene la cantidad
real de anuncios observados para una combinación de ciudad y tipo de alojamiento.

También se calculan porcentajes dentro de cada ciudad. Los conteos son necesarios
para la prueba estadística y los porcentajes facilitan la interpretación de negocio.

In [3]:
room_type_counts = pd.crosstab(
    analysis_data["city"],
    analysis_data["room_type"],
)

room_type_percentages = (
    pd.crosstab(
        analysis_data["city"],
        analysis_data["room_type"],
        normalize="index",
    )
    .mul(100)
    .round(2)
)

display(room_type_counts)
display(room_type_percentages)

room_type,Entire home/apt,Hotel room,Private room,Shared room
city,,,,
London,47445,1113,35882,628
Madrid,11314,166,7809,329
Milan,13605,74,4376,267
New York,25409,0,22326,1160
Sydney,22918,0,13115,629
Tokyo,7463,0,3004,999


room_type,Entire home/apt,Hotel room,Private room,Shared room
city,,,,
London,55.77,1.31,42.18,0.74
Madrid,57.67,0.85,39.81,1.68
Milan,74.25,0.40,23.88,1.46
New York,51.97,0.00,45.66,2.37
Sydney,62.51,0.00,35.77,1.72
Tokyo,65.09,0.00,26.20,8.71


### 4.2 Prueba chi-cuadrado y comprobación de supuestos

La prueba calcula las frecuencias que esperaríamos si la ciudad y el tipo de
alojamiento fueran independientes. Después compara esas frecuencias esperadas con
los conteos realmente observados.

Para interpretar el resultado se comprueba que:

- cada anuncio pertenece a una sola combinación de ciudad y tipo;
- ninguna frecuencia esperada sea menor que 1;
- como máximo el 20 % de las frecuencias esperadas sean menores que 5.

Los anuncios son unidades distintas, aunque la independencia es aproximada porque
un mismo anfitrión puede publicar varios anuncios.*

In [4]:
(
    chi_square_statistic,
    chi_square_p_value,
    degrees_of_freedom,
    expected_values,
) = chi2_contingency(room_type_counts)

expected_counts = pd.DataFrame(
    expected_values,
    index=room_type_counts.index,
    columns=room_type_counts.columns,
)

minimum_expected_count = expected_counts.min().min()
expected_below_five_percentage = (
    expected_counts.lt(5).to_numpy().mean() * 100
)

assumption_checks = pd.Series(
    {
        "minimum_expected_count": minimum_expected_count,
        "expected_below_five_percentage": expected_below_five_percentage,
    },
    name="value",
)

assert minimum_expected_count >= 1
assert expected_below_five_percentage <= 20

display(expected_counts.round(2))
assumption_checks.to_frame().round(2)

room_type,Entire home/apt,Hotel room,Private room,Shared room
city,,,,
London,49546.68,523.09,33447.12,1551.11
Madrid,11426.23,120.63,7713.42,357.71
Milan,10671.39,112.66,7203.86,334.08
New York,28478.21,300.66,19224.58,891.54
Sydney,21353.27,225.44,14414.80,668.49
Tokyo,6678.21,70.51,4508.21,209.07


,value
minimum_expected_count,70.51
expected_below_five_percentage,0.00


### 4.3 Significación y tamaño del efecto

El p-valor permite decidir si se rechaza la hipótesis de independencia, pero no
indica si la asociación es importante.

Por eso se calcula V de Cramér:

- 0 representa ausencia de asociación;
- los valores próximos a 1 representan una asociación fuerte.

Con una muestra muy grande, una diferencia pequeña puede producir un p-valor muy
bajo. La conclusión debe considerar conjuntamente el p-valor, V de Cramér y las
proporciones observadas.

In [5]:
total_listings = room_type_counts.to_numpy().sum()
row_count, column_count = room_type_counts.shape

cramers_v = np.sqrt(
    chi_square_statistic
    / (
        total_listings
        * min(row_count - 1, column_count - 1)
    )
)

chi_square_result = pd.Series(
    {
        "chi_square_statistic": chi_square_statistic,
        "degrees_of_freedom": degrees_of_freedom,
        "p_value": chi_square_p_value,
        "cramers_v": cramers_v,
        "reject_null_hypothesis": chi_square_p_value < ALPHA,
    },
    name="result",
)

display(chi_square_result.to_frame())
print(f"P-valor: {chi_square_p_value:.3e}")
print(f"V de Cramér: {cramers_v:.4f}")

,result
chi_square_statistic,8767.451221
degrees_of_freedom,15
p_value,0.0
cramers_v,0.115248
reject_null_hypothesis,True


P-valor: 0.000e+00
V de Cramér: 0.1152


### 4.4 Interpretación del resultado

La prueba chi-cuadrado encontró una asociación estadísticamente significativa entre
la ciudad y el tipo de alojamiento, `χ²(15) = 8767,45`, `p < 0,001`. Por tanto,
se rechaza la hipótesis nula de independencia.

El p-valor aparece como `0.0` por la precisión numérica del ordenador: no representa
una probabilidad literalmente igual a cero, sino un valor extremadamente pequeño.

Sin embargo, V de Cramér fue `0,1152`, lo que indica que la magnitud de la asociación
es pequeña. La elevada cantidad de anuncios permite detectar diferencias que no
necesariamente son grandes.

Las proporciones ayudan a entender el patrón: Milán presenta una proporción
especialmente alta de alojamientos completos; Tokio destaca por una mayor presencia
relativa de habitaciones compartidas; y las habitaciones de hotel solo aparecen en
Londres, Madrid y Milán dentro de estos archivos.

Para Airbnb, la ciudad es una dimensión útil para explorar la composición de la
oferta, pero no debe considerarse por sí sola un predictor fuerte del tipo de
alojamiento. El resultado muestra asociación, no causalidad, y la independencia
entre observaciones es aproximada porque un anfitrión puede publicar varios anuncios.

## 5. Hipótesis 2: estancia mínima y actividad aproximada

### 5.1 Preparación de la actividad

`reviews_per_month` se utiliza como indicador aproximado de actividad:

- si un anuncio no tiene ninguna reseña y su tasa mensual está vacía, se asigna
  actividad cero;
- si tiene reseñas acumuladas pero la tasa mensual está vacía, la actividad se
  considera desconocida y el anuncio se excluye de este contraste.

Esta regla no modifica los datos originales. Crea una variable analítica separada
y evita convertir información contradictoria en un cero artificial.

In [6]:
analysis_data["review_activity_proxy"] = analysis_data["reviews_per_month"]

no_reviews_without_rate = (
    analysis_data["reviews_per_month"].isna()
    & analysis_data["number_of_reviews"].eq(0)
)

reviews_without_monthly_rate = (
    analysis_data["reviews_per_month"].isna()
    & analysis_data["number_of_reviews"].gt(0)
)

analysis_data.loc[
    no_reviews_without_rate,
    "review_activity_proxy",
] = 0

analysis_data["minimum_nights_group"] = np.where(
    analysis_data["minimum_nights"].gt(8),
    "More than 8 nights",
    "8 nights or fewer",
)

activity_data = analysis_data.loc[
    ~reviews_without_monthly_rate
].copy()

activity_quality_checks = pd.Series(
    {
        "total_listings": len(analysis_data),
        "unknown_activity_count": reviews_without_monthly_rate.sum(),
        "eligible_listing_count": len(activity_data),
        "missing_activity_after_exclusion": (
            activity_data["review_activity_proxy"].isna().sum()
        ),
    },
    name="value",
)

assert activity_quality_checks["unknown_activity_count"] == 123
assert activity_quality_checks["missing_activity_after_exclusion"] == 0

activity_quality_checks.to_frame()

,value
total_listings,220031
unknown_activity_count,123
eligible_listing_count,219908
missing_activity_after_exclusion,0


### 5.2 Descripción de los grupos

Antes de ejecutar la prueba se muestra el tamaño, la mediana y los cuartiles de
cada grupo. Esto permite interpretar la dirección y la magnitud del patrón sin
depender únicamente del p-valor.

In [7]:
activity_group_summary = (
    activity_data
    .groupby(
        ["city", "minimum_nights_group"],
        as_index=False,
    )
    .agg(
        listing_count=("id", "size"),
        median_activity=("review_activity_proxy", "median"),
        first_quartile=("review_activity_proxy", lambda values: values.quantile(0.25)),
        third_quartile=("review_activity_proxy", lambda values: values.quantile(0.75)),
    )
    .sort_values(["city", "minimum_nights_group"])
)

activity_group_summary

,city,minimum_nights_group,listing_count,median_activity,first_quartile,third_quartile
0,London,8 nights or fewer,80998,0.43,0.04,1.3000
1,London,More than 8 nights,4070,0.07,0.00,0.3800
2,Madrid,8 nights or fewer,17621,0.27,0.00,1.1700
3,Madrid,More than 8 nights,1997,0.04,0.00,0.3100
4,Milan,8 nights or fewer,16963,0.14,0.00,0.6600
5,Milan,More than 8 nights,1359,0.01,0.00,0.2100
6,New York,8 nights or fewer,41692,0.51,0.06,1.8900
7,New York,More than 8 nights,7203,0.09,0.00,0.3300
8,Sydney,8 nights or fewer,33505,0.18,0.00,1.0000
9,Sydney,More than 8 nights,3034,0.00,0.00,0.1100


### 5.3 Mann–Whitney, tamaño del efecto y corrección de Holm

La prueba se ejecuta por separado para cada ciudad. La correlación biserial por
rangos indica la dirección del resultado:

- valor negativo: el grupo de más de ocho noches tiende a menor actividad;
- valor positivo: tiende a mayor actividad;
- valor próximo a cero: la diferencia práctica es reducida.

La corrección de Holm ajusta los seis p-valores para controlar la multiplicidad.

In [8]:
def holm_adjust(p_values):
    """Ajusta una familia de p-valores mediante el método de Holm."""
    p_values = np.asarray(p_values, dtype=float)
    test_count = len(p_values)
    order = np.argsort(p_values)
    ordered_p_values = p_values[order]

    ordered_adjusted = np.maximum.accumulate(
        (test_count - np.arange(test_count)) * ordered_p_values
    )
    ordered_adjusted = np.minimum(ordered_adjusted, 1)

    adjusted = np.empty(test_count)
    adjusted[order] = ordered_adjusted

    return adjusted


mann_whitney_results = []

for city in sorted(activity_data["city"].unique()):
    city_data = activity_data.loc[activity_data["city"].eq(city)]

    reference_activity = city_data.loc[
        city_data["minimum_nights_group"].eq("8 nights or fewer"),
        "review_activity_proxy",
    ]

    long_stay_activity = city_data.loc[
        city_data["minimum_nights_group"].eq("More than 8 nights"),
        "review_activity_proxy",
    ]

    assert len(reference_activity) > 0
    assert len(long_stay_activity) > 0

    test_result = mannwhitneyu(
        long_stay_activity,
        reference_activity,
        alternative="two-sided",
        method="asymptotic",
    )

    rank_biserial = (
        2 * test_result.statistic
        / (len(long_stay_activity) * len(reference_activity))
        - 1
    )

    mann_whitney_results.append(
        {
            "city": city,
            "reference_count": len(reference_activity),
            "long_stay_count": len(long_stay_activity),
            "u_statistic": test_result.statistic,
            "p_value": test_result.pvalue,
            "rank_biserial": rank_biserial,
        }
    )

mann_whitney_results = pd.DataFrame(mann_whitney_results)

mann_whitney_results["adjusted_p_value"] = holm_adjust(
    mann_whitney_results["p_value"]
)

mann_whitney_results["reject_null_hypothesis"] = (
    mann_whitney_results["adjusted_p_value"] < ALPHA
)

mann_whitney_results[
    [
        "city",
        "reference_count",
        "long_stay_count",
        "p_value",
        "adjusted_p_value",
        "rank_biserial",
        "reject_null_hypothesis",
    ]
]

,city,reference_count,long_stay_count,p_value,adjusted_p_value,rank_biserial,reject_null_hypothesis
0,London,80998,4070,9.555142e-306,4.777571e-305,-0.344378,True
1,Madrid,17621,1997,2.755517e-101,8.266550e-101,-0.287793,True
2,Milan,16963,1359,3.237020e-75,3.237020e-75,-0.295518,True
3,New York,41692,7203,0.000000e+00,0.000000e+00,-0.405775,True
4,Sydney,33505,3034,4.125331e-283,1.650132e-282,-0.386852,True
5,Tokyo,10746,720,1.779028e-85,3.558055e-85,-0.434798,True


In [9]:
def classify_effect(effect):
    absolute_effect = abs(effect)

    if absolute_effect < 0.10:
        return "Negligible"
    if absolute_effect < 0.30:
        return "Small"
    if absolute_effect < 0.50:
        return "Moderate"
    return "Large"


median_comparison = (
    activity_group_summary
    .pivot(
        index="city",
        columns="minimum_nights_group",
        values="median_activity",
    )
    .reset_index()
    .rename(
        columns={
            "8 nights or fewer": "reference_median_activity",
            "More than 8 nights": "long_stay_median_activity",
        }
    )
)

interpretable_results = (
    mann_whitney_results
    .merge(
        median_comparison,
        on="city",
        how="left",
        validate="one_to_one",
    )
)

interpretable_results["effect_magnitude"] = (
    interpretable_results["rank_biserial"].apply(classify_effect)
)

interpretable_results[
    [
        "city",
        "reference_median_activity",
        "long_stay_median_activity",
        "rank_biserial",
        "effect_magnitude",
        "reject_null_hypothesis",
    ]
].round(3)

,city,reference_median_activity,long_stay_median_activity,rank_biserial,effect_magnitude,reject_null_hypothesis
0,London,0.43,0.07,-0.344,Moderate,True
1,Madrid,0.27,0.04,-0.288,Small,True
2,Milan,0.14,0.01,-0.296,Small,True
3,New York,0.51,0.09,-0.406,Moderate,True
4,Sydney,0.18,0.00,-0.387,Moderate,True
5,Tokyo,1.83,0.43,-0.435,Moderate,True


### 5.4 Interpretación del resultado

Después de aplicar la corrección de Holm, las seis ciudades presentan un p-valor
ajustado inferior a 0,05. Por tanto, se rechaza H0 en todas ellas: la distribución
de actividad aproximada difiere entre los anuncios con estancias mínimas de hasta
ocho noches y los que exigen más de ocho.

Todos los tamaños del efecto son negativos. Esto indica que los anuncios con una
estancia mínima superior a ocho noches tienden a presentar menor actividad mensual
de reseñas.

La magnitud es pequeña en Madrid (`r = −0,288`) y Milán (`r = −0,296`), y moderada
en Londres (`r = −0,344`), Sídney (`r = −0,387`), Nueva York (`r = −0,406`) y
Tokio (`r = −0,435`).

Los descriptivos muestran la misma dirección:

- Londres: la mediana disminuye de 0,43 a 0,07.
- Madrid: disminuye de 0,27 a 0,04.
- Milán: disminuye de 0,14 a 0,01.
- Nueva York: disminuye de 0,51 a 0,09.
- Sídney: disminuye de 0,18 a 0,00.
- Tokio: disminuye de 1,83 a 0,43.

Los p-valores extremadamente pequeños están influidos por el gran tamaño de la
muestra. Los tamaños del efecto muestran que las diferencias no tienen la misma
intensidad en todas las ciudades.

Para Airbnb, los anuncios con restricciones superiores a ocho noches constituyen
un segmento que merece una revisión diferenciada. Podría investigarse si representan
alquileres orientados a estancias largas, restricciones regulatorias, decisiones
operativas o anuncios con menor dinamismo.

Este análisis no demuestra que aumentar la estancia mínima provoque menor actividad.
También pueden influir el barrio, el tipo de alojamiento, la antigüedad del anuncio,
las características de la propiedad y las políticas locales.

## 6. Conclusiones y limitaciones

### Conclusiones

1. La ciudad y el tipo de alojamiento presentan una asociación estadísticamente
   significativa, pero su magnitud es pequeña (`V de Cramér = 0,1152`). Esto respalda
   la segmentación por ciudad sin considerar que la ciudad determine fuertemente el
   tipo de oferta.

2. En las seis ciudades, los anuncios con estancias mínimas superiores a ocho noches
   presentan menor actividad aproximada de reseñas. La magnitud de la diferencia es
   pequeña o moderada según la ciudad.

3. La significación estadística no sustituye la relevancia práctica. El tamaño del
   efecto es indispensable porque la gran cantidad de anuncios puede producir
   p-valores muy pequeños incluso ante diferencias limitadas.

### Limitaciones

- `reviews_per_month` es una aproximación de actividad, no una medición de reservas,
  demanda u ocupación.
- Se excluyeron 123 anuncios con reseñas acumuladas pero sin tasa mensual porque su
  actividad no puede interpretarse de manera fiable.
- El umbral de ocho noches es una decisión analítica orientada a detectar restricciones
  superiores a una semana, no un límite oficial de Airbnb.
- Los datos representan una fotografía sin fecha de extracción documentada.
- La antigüedad del anuncio y otras características relevantes no están controladas.
- Varios anuncios pueden pertenecer al mismo anfitrión, por lo que la independencia
  entre observaciones es aproximada.
- Las pruebas detectan asociaciones y diferencias observadas, pero no demuestran
  relaciones causales.